In [1]:
# -----------------------------------------------------------------------------
# NOTEBOOK EXECUTION CONTROL
# -----------------------------------------------------------------------------
# Set this to True to run individual component tests sequentially
RUN_DEBUG_STEPS = True


In [ ]:
import sys
import os
import logging
import json
import dataclasses

# -----------------------------------------------------------------------------
# PATH SETUP: Point to your project root to load 'common' and 'microservices'
# -----------------------------------------------------------------------------
project_root = os.path.abspath('../../..') 
if project_root not in sys.path:
    sys.path.append(project_root)

# -----------------------------------------------------------------------------
# IMPORT EXISTING BACKEND DATA STRUCTURES
# -----------------------------------------------------------------------------
try:
    from common.models.api.redis_models import (
        Article, NLPResult, NLPOptions, Claim, Entity, SentenceScore, BiasProfile
    )
    from microservices.nlp.models.base import SentenceProcessor, ArticleProcessor
    print("Successfully loaded backend data structures.")
except ImportError as e:
    print(f"Import Failed: {e}")
    print("Ensure 'project_root' correctly points to the folder containing 'common/' and 'microservices/'")

# Configure Logging
logging.basicConfig(level=logging.INFO, format='%(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("Notebook")

Successfully loaded backend data structures.


In [3]:
import json
import os

# 1. Load data from article.json (populated from the RTF content)
# We assume article.json is in the same directory as this notebook
json_path = 'article3.json' 

if not os.path.exists(json_path):
    print(f"Error: '{json_path}' not found in the current directory.")
    # Stop execution if file is missing
    raise FileNotFoundError(f"Please ensure {json_path} is next to this notebook.")

with open(json_path, 'r') as f:
    data = json.load(f)
    print(f"Successfully loaded '{json_path}'")

# 2. Create Article Object (using backend model)
# We map keys from the updated JSON format to the Article attributes
article = Article(
    title=data.get('article_title', 'Unknown Title'),
    text=data.get('article_text', ''),
    # Map 'article_url' from JSON to the Article's 'link' field
    link=data.get('article_url', ''),
    summary=data.get('article_summary', '')
)

# 3. Initialize Result and Options
result = NLPResult()
options = NLPOptions(min_confidence=0.8)

print(f"Initialized Article: {article.title}")

Successfully loaded 'article3.json'
Initialized Article: FBI raids Georgia election office over 2020 voter fraud claims


## Preprocessor

In [ ]:
import logging
import spacy
import re
from typing import List

# Local imports
from microservices.nlp.models.base import SentenceProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore
from microservices.nlp.config import PREPROCESS_MIN_TOKENS, PHOTO_CREDIT_MAX_LEN

logger = logging.getLogger(__name__)

class Preprocessor(SentenceProcessor):
    """
    "Universal Janitor" Preprocessor.

    Strategies applied:
    1. Regex Cleaning: Removes distinct artifacts (Dates, UI buttons, Footers) using strict patterns.
    2. Footer Cutoff: Stops processing the text entirely once footer keywords are detected.
    3. Photo Credit Filter: Drops slash-separated agency attribution lines (line-level and
       sentence-level), guarded by a verb check to preserve inline agency mentions.
    4. Linguistic Filtering: Uses Spacy's POS tagger to remove short lines (< PREPROCESS_MIN_TOKENS
       tokens) that lack verbs (e.g., "Politics", "Frank Gardner").

    Returns sentences via a local list; does NOT write to result.sentences.
    """
    def __init__(self, nlp=None):
        if nlp is not None:
            logger.info("Preprocessor: Using shared spaCy model.")
            self.nlp = nlp
        else:
            logger.info("Preprocessor: Loading Spacy 'en_core_web_sm' model...")
            try:
                self.nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
            except OSError:
                logger.error("Spacy model not found. Run: python -m spacy download en_core_web_sm")
                raise

        self._photo_credit_re = re.compile(
            r'(?i)\b(getty|reuters|afp|ntb|epa|ugc|ap|bbc|pool|handout|'
            r'shutterstock|alamy|corbis|zuma|sipa|nurphoto|xinhua|'
            r'press association|pa images|sky news|itv|abc news)\b'
        )

    def _clean_and_repair_structure(self, raw_text: str) -> str:
        if not raw_text:
            return ""

        lines = raw_text.split('\n')
        cleaned_lines = []

        footer_cutoff_pattern = re.compile(r'(?i)^('
            r'more from (the )?bbc|related (content|stories|topics)|up next|most popular|'
            r'have you read\?|more on geographies|license and republishing|content index|'
            r'bbc\.com help|privacy policy|about us|follow .* on|sign up for'
        r')')

        time_meta_pattern = re.compile(r'(?i)^('
            r'\d+\s+(hour|minute|day|second|hr|min)s?\s+ago|'
            r'updated\s+.*|'
            r'\d+\s+min\s+read|'
            r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{1,2},?\s+\d{4}'
        r')')

        credits_pattern = re.compile(r'(?i)^('
            r'(image|photo|source|graphic|credits?):|'
            r'(left|right|top|bottom):|'
            r'analysis by|'
            r'this article is part of:|'
            r'unknown\.|'
            r'getty images|epa|afp|ugc|reuters|ap|copyright|davidoff studios'
        r')')

        ui_pattern = re.compile(r'(?i)^('
            r'register|sign in|log in|'
            r'skip to.*|'
            r'share|save|follow|subscribe|'
            r'menu|home|news|sport|weather|'
            r'listen to .* read this article|'
            r'loading\.\.\.|'
            r'create a free account|'
            r'terms of use'
        r')')

        byline_pattern = re.compile(r'(?i)^('
            r'by\s+[A-Z][a-z]+\s+[A-Z][a-z]+|'
            r'.*correspondent.*|'
            r'writer,.*'
        r')')

        photo_credit_pattern = self._photo_credit_re

        for line in lines:
            line = line.strip()
            if not line:
                continue
            if len(line) < 4:
                continue
            if footer_cutoff_pattern.search(line):
                break
            if ui_pattern.search(line):
                continue
            if time_meta_pattern.search(line):
                continue
            if credits_pattern.search(line):
                continue
            if '/' in line and len(line) < PHOTO_CREDIT_MAX_LEN and photo_credit_pattern.search(line):
                has_verb = any(t.pos_ in ("VERB", "AUX") for t in self.nlp(line))
                if not has_verb:
                    continue
            if byline_pattern.search(line) and len(line) < 50:
                continue
            if line[-1] not in ".?!:;\"'":
                line += "."
            cleaned_lines.append(line)

        return " ".join(cleaned_lines)

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> List[SentenceScore]:
        """
        Cleans and tokenizes the article text into local SentenceScore objects.
        """
        raw_text = getattr(article, 'text', getattr(article, 'content', ""))
        clean_text = self._clean_and_repair_structure(raw_text)

        if not clean_text:
            logger.warning("Preprocessor: Text was empty after cleaning.")
            return []

        doc = self.nlp(clean_text)

        sentence_objects = []
        for idx, span in enumerate(doc.sents):
            text_segment = span.text.strip()
            token_count = len(span)

            if token_count < PREPROCESS_MIN_TOKENS:
                if "?" in text_segment:
                    pass
                else:
                    has_verb = any(token.pos_ in ["VERB", "AUX"] for token in span)
                    if not has_verb:
                        continue

            # Sentence-level photo credit guard
            if ('/' in text_segment
                    and len(text_segment) < PHOTO_CREDIT_MAX_LEN
                    and self._photo_credit_re.search(text_segment)
                    and not any(tok.pos_ in ("VERB", "AUX") for tok in span)):
                continue

            s_obj = SentenceScore(
                index=idx,
                text=text_segment,
                score=0.0,
                embedding=None
            )
            sentence_objects.append(s_obj)

        logger.info(f"Preprocessor: Cleaned & Split. Result: {len(sentence_objects)} sentences.")
        return sentence_objects

if RUN_DEBUG_STEPS:
    print("--- Running Preprocessor Test ---")
    try:
        pre = Preprocessor()
        sentences = pre.run(article, result, options)

        print(f"Success! Split into {len(sentences)} sentences.")
        for s in sentences:
            print(f"  [{s.index}] {s.text}")

    except Exception as e:
        print(f"Error: {e}")

__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Running Preprocessor Test ---


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 26 sentences.


Success! Split into 26 sentences.
  [0] FBI raids Georgia election office over 2020 voter fraud claims.
  [1] The FBI raided a Georgia election office on Wednesday as it examined allegations of voter fraud in the 2020 US election.
  [2] In a statement to Reuters, the FBI said it was conducting a "court-authorised law enforcement activity" at the Fulton County Election Hub.
  [3] Fulton County officials said that the government's warrant "sought a number of records related to 2020 elections".
  [4] Donald Trump lost the state and the county - the most populated in Georgia - to Joe Biden in the 2020 election and has long said his loss was due to fraud, a claim that is unsubstantiated.
  [5] The Department of Justice (DOJ) sued Fulton County officials in December, seeking election-related materials from 2020.
  [6] The FBI and Fulton County did not immediately respond to the BBC's requests for comment.
  [7] Agents with FBI vests were seen entering and exiting the election office on Wedne

## Entity Recognizer

In [ ]:
import logging
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from typing import List, Dict, Any

# Local imports
from microservices.nlp.models.base import ArticleProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, Entity, SentenceScore
from microservices.nlp.config import NER_MODEL, NER_BATCH_SIZE

logger = logging.getLogger(__name__)

class EntityRecognizer(ArticleProcessor):
    """
    Named Entity Recognition using the model specified in config.NER_MODEL.

    Processes the sentences list and writes deduplicated entities
    into result.entities_in_article.
    """

    def __init__(self):
        self.model_name = NER_MODEL
        self.device = 0 if torch.cuda.is_available() else -1
        use_fp16 = torch.cuda.is_available()

        logger.info(f"EntityRecognizer: Loading '{self.model_name}' "
                    f"on {'CUDA' if self.device == 0 else 'CPU'} "
                    f"(fp16={use_fp16})...")
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForTokenClassification.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16 if use_fp16 else torch.float32,
            )
            self.nlp_pipeline = pipeline(
                "ner",
                model=self.model,
                tokenizer=self.tokenizer,
                aggregation_strategy="simple",
                device=self.device,
                batch_size=NER_BATCH_SIZE,
            )
        except Exception as e:
            logger.error(f"EntityRecognizer: Failed to load model: {e}")
            raise

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> None:
        """
        Runs NER over all sentences provided by the Preprocessor.
        Updates result.entities_in_article with unique entities found.
        Character offsets stored in Entity are article-relative.
        """
        if not sentences:
            logger.warning("EntityRecognizer: No sentences provided to process.")
            return

        valid_sentences: List[SentenceScore] = [
            s for s in sentences if s.text and s.text.strip()
        ]
        if not valid_sentences:
            logger.warning("EntityRecognizer: All sentence texts are empty, skipping.")
            return

        texts: List[str] = [s.text for s in valid_sentences]
        logger.info(f"EntityRecognizer: Running NER on {len(texts)} sentences.")

        # Build cumulative char offsets so entity positions become article-relative
        char_offsets: List[int] = []
        offset = 0
        for t in texts:
            char_offsets.append(offset)
            offset += len(t) + 1  # +1 for the sentence separator

        unique_entities: Dict[tuple, Dict[str, Any]] = {}

        try:
            for sent_idx, sent_results in enumerate(self.nlp_pipeline(texts)):
                sent_offset = char_offsets[sent_idx]
                for item in sent_results:
                    text = item["word"].strip()
                    label = item["entity_group"]
                    score = float(item["score"])

                    if len(text) < 3:
                        continue

                    key = (text.lower(), label)
                    if key not in unique_entities or score > unique_entities[key]["score"]:
                        e_obj = Entity(
                            entity_text=text,
                            type_of_entity=label,
                            start_char=sent_offset + item.get("start", 0),
                            end_char=sent_offset + item.get("end", 0),
                        )
                        unique_entities[key] = {"entity_obj": e_obj, "score": score}

            result.entities_in_article = [v["entity_obj"] for v in unique_entities.values()]
            logger.info(f"EntityRecognizer: Found {len(result.entities_in_article)} unique entities.")

        except Exception as e:
            logger.error(f"EntityRecognizer failed during execution: {e}")
            raise

if RUN_DEBUG_STEPS:
    print("--- Running Entity Recognizer Test ---")
    print(f"Input: {len(sentences)} sentences from Preprocessor")
    try:
        ner = EntityRecognizer()
        ner.run(article, result, options, sentences)

        print(f"Found {len(result.entities_in_article)} unique entities.")
        for e in result.entities_in_article:
            print(f"  - {e.entity_text} ({e.type_of_entity})")

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()

/home/farhan/miniconda2/envs/nlp311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...


--- Running Entity Recognizer Test ---
Input: 26 sentences from Preprocessor


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 140.83it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
__main__ - INFO - EntityRecognizer: Running NER on 26 sentences.
__main__ - INFO - EntityRecognizer: Found 20 unique entities.


Found 20 unique entities.
  - fbi (ORG)
  - georgia (LOC)
  - reuters (ORG)
  - fulton county (LOC)
  - fulton county (ORG)
  - donald trump (PER)
  - joe biden (PER)
  - department of justice (ORG)
  - doj (ORG)
  - bbc (ORG)
  - mo ivory (PER)
  - democrat (MISC)
  - bill clinton (PER)
  - biden (PER)
  - trump (PER)
  - raffen (PER)
  - ##sperger (ORG)
  - white house (LOC)
  - trump (ORG)
  - georgian (MISC)


## Sentence Extraction + Deduplication

In [ ]:
import logging
import torch
import numpy as np
import torch.nn.functional as F
from typing import List
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
from torch.amp import autocast

# Local imports
from microservices.nlp.models.base import SentenceProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore
from microservices.nlp.config import (
    BERT_SCORING_MODEL, NLI_MODEL,
    SENTENCE_SCORING_BATCH, NLI_MAX_PAIRS,
    NLI_ENTAILMENT_THRESHOLD, BERT_MAX_LENGTH,
    SENTENCE_EXTRACT_TOP_K,
)

logger = logging.getLogger(__name__)

class SentenceExtraction(SentenceProcessor):
    """
    MODULAR SENTENCE EXTRACTION LAYER
    Combines extractive importance (BertSum-style CLS scoring) with
    logical deduplication (NLI cross-encoder).

    Accepts and returns a local sentences list; does NOT touch result.
    """
    def __init__(self, use_fp16: bool = True):
        self.use_fp16 = use_fp16 and torch.cuda.is_available()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        logger.info(f"SentenceExtraction: Initializing models on {self.device} "
                    f"(fp16={self.use_fp16})...")

        try:
            self.tokenizer = AutoTokenizer.from_pretrained(BERT_SCORING_MODEL)
            self.scoring_model = AutoModel.from_pretrained(BERT_SCORING_MODEL).to(self.device)

            self.nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
            self.nli_model = (
                AutoModelForSequenceClassification
                .from_pretrained(NLI_MODEL)
                .to(self.device)
            )

            if self.use_fp16:
                self.scoring_model = self.scoring_model.half()
                self.nli_model = self.nli_model.half()
        except Exception as e:
            logger.error(f"SentenceExtraction: Failed to load models: {e}")
            raise

    def _get_salience_scores(self, sentences: List[str]) -> np.ndarray:
        scores = []
        with torch.no_grad():
            with autocast(device_type=("cuda" if "cuda" in self.device else "cpu"), enabled=self.use_fp16):
                for start in range(0, len(sentences), SENTENCE_SCORING_BATCH):
                    batch_texts = sentences[start : start + SENTENCE_SCORING_BATCH]
                    inputs = self.tokenizer(
                        batch_texts,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=BERT_MAX_LENGTH,
                    ).to(self.device)
                    outputs = self.scoring_model(**inputs)
                    cls_vecs = outputs.last_hidden_state[:, 0, :]
                    batch_scores = torch.mean(torch.abs(cls_vecs), dim=-1)
                    scores.extend(batch_scores.cpu().float().tolist())

        scores = np.array(scores)
        return (
            (scores - scores.min()) / (scores.max() - scores.min() + 1e-10)
            if len(scores) > 1
            else scores
        )

    def _is_redundant(self, candidate: str, already_selected: List[str]) -> bool:
        if not already_selected:
            return False

        pairs = [[prev, candidate] for prev in already_selected]
        pairs = pairs[-NLI_MAX_PAIRS:]

        with torch.no_grad():
            with autocast(device_type=("cuda" if "cuda" in self.device else "cpu"), enabled=self.use_fp16):
                inputs = self.nli_tokenizer(
                    pairs,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=BERT_MAX_LENGTH,
                ).to(self.device)
                outputs = self.nli_model(**inputs)
                probs = F.softmax(outputs.logits.float(), dim=-1)
                return bool((probs[:, 1] > NLI_ENTAILMENT_THRESHOLD).any())

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Scores and deduplicates sentences.
        Returns a filtered local list of high-value sentences.
        """
        if not sentences:
            return []

        raw_texts = [s.text for s in sentences]
        scores = self._get_salience_scores(raw_texts)

        ranked_indices = np.argsort(scores)[::-1]
        selected_indices: List[int] = []
        selected_texts:   List[str] = []

        limit = getattr(options, "max_claims", SENTENCE_EXTRACT_TOP_K)

        for idx in ranked_indices:
            if len(selected_indices) >= limit:
                break
            candidate = raw_texts[idx]
            if not self._is_redundant(candidate, selected_texts):
                selected_indices.append(int(idx))
                selected_texts.append(candidate)
                sentences[idx].score = float(scores[idx])

        extracted = [sentences[i] for i in sorted(selected_indices)]
        logger.info(f"SentenceExtraction: Extracted {len(extracted)} salient, unique sentences.")
        return extracted

if RUN_DEBUG_STEPS:
    print("--- Running Sentence Extraction Test ---")
    try:
        extractor = SentenceExtraction(use_fp16=True)
        sentences = extractor.run(article, result, options, sentences)

        print(f"Success! Extracted {len(sentences)} sentences.")
        print("Top 5 Extracted Sentences:")
        for i, s in enumerate(sentences[:5]):
            print(f"  [{s.index}] (Score: {s.score:.4f}) {s.text[:100]}...")

    except Exception as e:
        print(f"Error: {e}")

In [7]:
if RUN_DEBUG_STEPS:
    print("--- Running Sentence Extraction Test ---")
    try:
        extractor = SentenceExtraction(use_fp16=True)
        sentences = extractor.run(article, result, options, sentences)
        
        print(f"Success! Extracted {len(sentences)} sentences.")
        print("Top 5 Extracted Sentences:")
        for i, s in enumerate(sentences[:5]):
            print(f"  [{s.index}] (Score: {s.score:.4f}) {s.text[:100]}...")
            
    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - SentenceExtraction: Initializing models on cuda (fp16=True)...


--- Running Sentence Extraction Test ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 304.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 247.72it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]   

Success! Extracted 10 sentences.
Top 5 Extracted Sentences:
  [0] (Score: 1.0000) FBI raids Georgia election office over 2020 voter fraud claims....
  [1] (Score: 0.6579) The FBI raided a Georgia election office on Wednesday as it examined allegations of voter fraud in t...
  [9] (Score: 0.5770) "This is an assault on your vote," Fulton County Commissioner Mo Ivory said at a press conference ou...
  [11] (Score: 0.7715) The 2020 election marked the first time since 1992 that a Democrat had won the southern US state of ...
  [17] (Score: 0.6241) The state of Georgia, and Fulton County in particular, were a major focus of Trump's efforts to over...


## Decontextualizer

In [ ]:
import torch
import logging
import spacy
import re
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from typing import List, Optional

# Local imports
from microservices.nlp.models.base import SentenceProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore
from microservices.nlp.config import (
    QG_MODEL, QA_MODEL, GEN_MODEL,
    BM25_TOP_K, QA_SCORE_THRESHOLD,
    BERT_MAX_LENGTH, DECONTEXT_MAX_GEN_LENGTH,
    DECONTEXT_MAX_UNITS, DECONTEXT_REWRITE_RATIO,
    DECONTEXT_GEN_BATCH_SIZE,
)

logger = logging.getLogger(__name__)

class Decontextualizer(SentenceProcessor):
    """
    MODULAR DECONTEXTUALIZER LAYER — fully batched inference.

    All four model passes (QG / QA / QA2D / rewrite) are batched across
    every sentence in a single forward sweep instead of looping per-sentence.

    1.  Unit Extraction (spaCy): named entities, pronouns, noun chunks, root verb phrases.
    2.  Question Generation (QG_MODEL): one question per ambiguous unit.
    3.  BM25 Evidence Retrieval: grounds each question against the full article text.
    4.  QA Grounding (QA_MODEL): answers each question; answers below QA_SCORE_THRESHOLD discarded.
    5.  QA-to-Declarative (GEN_MODEL): Q-A pairs converted to declarative context sentences.
    6.  Final Rewrite (GEN_MODEL): self-contained rewrite incorporating resolved context.
    7.  Quality Gate: reject rewrite if empty, unchanged, contains '?', or >DECONTEXT_REWRITE_RATIO× original length.

    Accepts and returns a local sentences list; does NOT write to result.
    """

    _LABEL_RE = re.compile(
        r"^(false|true|fake|entailment|neutral|contradiction)\b[\s:,]*",
        flags=re.IGNORECASE,
    )

    def __init__(self, use_gpu: bool = True, nlp=None):
        self.device    = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.device_id = 0 if self.device == "cuda" else -1
        use_fp16       = self.device == "cuda"

        logger.info(f"Decontextualizer: Initializing on {self.device} (fp16={use_fp16})...")

        if nlp is not None:
            logger.info("Decontextualizer: Using shared spaCy model.")
            self.nlp = nlp
        else:
            try:
                self.nlp = spacy.load("en_core_web_sm")
            except OSError:
                logger.error("Decontextualizer: Run: python -m spacy download en_core_web_sm")
                raise

        self.qg_tokenizer = AutoTokenizer.from_pretrained(QG_MODEL)
        self.qg_model = AutoModelForSeq2SeqLM.from_pretrained(
            QG_MODEL,
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        ).to(self.device)

        self.qa_pipe = pipeline(
            "question-answering",
            model=QA_MODEL,
            device=self.device_id,
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        )

        self.gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(
            GEN_MODEL,
            torch_dtype=torch.float16 if use_fp16 else torch.float32,
        ).to(self.device)

        logger.info("Decontextualizer: All models loaded successfully.")

    def _sanitize(self, text: str) -> str:
        return self._LABEL_RE.sub("", text).strip()

    def _generate_batch(
        self,
        prompts: List[str],
        tokenizer,
        model,
        batch_size: int = 8,
    ) -> List[str]:
        """
        Chunked batched seq2seq generation with beam search.
        Processes prompts in chunks of `batch_size` to avoid GPU OOM on large batches.
        """
        if not prompts:
            return []
        results: List[str] = []
        for chunk_start in range(0, len(prompts), batch_size):
            chunk = prompts[chunk_start : chunk_start + batch_size]
            inputs = tokenizer(
                chunk, return_tensors="pt", padding=True, truncation=True,
                max_length=BERT_MAX_LENGTH,
            ).to(self.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_length=DECONTEXT_MAX_GEN_LENGTH, num_beams=4,
                    repetition_penalty=2.5, early_stopping=True,
                )
            results.extend(
                self._sanitize(tokenizer.decode(o, skip_special_tokens=True))
                for o in outputs
            )
        return results

    def _extract_units(self, doc) -> List[str]:
        units: List[str] = []
        for ent in doc.ents:
            units.append(ent.text)
        for token in doc:
            if token.pos_ == "PRON":
                units.append(token.text)
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) < 5:
                units.append(chunk.text)
        for token in doc:
            if token.pos_ == "VERB" and token.dep_ in ("ROOT", "relcl", "advcl", "xcomp"):
                phrase_tokens = [token] + [
                    c for c in token.children
                    if c.dep_ in ("dobj", "prt", "attr") and c.i < token.i + 4
                ]
                phrase_tokens.sort(key=lambda t: t.i)
                phrase = " ".join(t.text for t in phrase_tokens)
                if phrase:
                    units.append(phrase)
        return [u for u in set(units) if len(u.split()) < 6 and len(u) > 1]

    def _bm25_retrieve(self, query: str, doc_sentences: List[str]) -> str:
        if not doc_sentences:
            return ""
        tokenized_corpus = [s.lower().split() for s in doc_sentences]
        bm25 = BM25Okapi(tokenized_corpus)
        scores = bm25.get_scores(query.lower().split())
        top_k = min(BM25_TOP_K, len(doc_sentences))
        top_indices = scores.argsort()[-top_k:][::-1]
        return " ".join(doc_sentences[i] for i in sorted(top_indices))

    def run(
        self,
        article: Article,
        result: NLPResult,
        options: NLPOptions,
        sentences: List[SentenceScore],
    ) -> List[SentenceScore]:
        """
        Rewrites each sentence to be self-contained using fully batched inference.

        All four model passes (QG / QA / QA2D / rewrite) are batched across
        every sentence in a single forward sweep instead of looping per-sentence.
        """
        if not sentences:
            return []

        n = len(sentences)

        # BM25 corpus: all article sentences used as evidence for QA
        doc_sentences = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", article.text or "")
            if len(s.strip()) > 10
        ]

        # ── Phase 1: batch spaCy parse ──────────────────────────────────────
        texts = [self._sanitize(s.text) for s in sentences]
        docs  = list(self.nlp.pipe(texts))

        # ── Phase 2: extract units; build flat QG prompt list ───────────────
        all_qg_prompts: List[str]   = []
        sent_qg_slices: List[tuple] = []   # (start, end) into all_qg_prompts

        for text, doc in zip(texts, docs):
            start = len(all_qg_prompts)
            if text:
                units = [
                    u for u in self._extract_units(doc) if len(u.split()) < 6
                ][:DECONTEXT_MAX_UNITS]
                all_qg_prompts.extend(f"answer: {u} context: {text}" for u in units)
            sent_qg_slices.append((start, len(all_qg_prompts)))

        # ── Phase 3: batch Question Generation ──────────────────────────────
        logger.info(
            f"Decontextualizer: QG batch — {len(all_qg_prompts)} prompts "
            f"across {n} sentences..."
        )
        all_questions = self._generate_batch(
            all_qg_prompts, self.qg_tokenizer, self.qg_model,
            batch_size=DECONTEXT_GEN_BATCH_SIZE,
        )

        # ── Phase 4: BM25 retrieval → flat QA input list ────────────────────
        all_qa_inputs: List[dict]   = []
        sent_qa_slices: List[tuple] = []   # (start, end) into all_qa_inputs

        for i, (qg_start, qg_end) in enumerate(sent_qg_slices):
            qa_start = len(all_qa_inputs)
            for question in all_questions[qg_start:qg_end]:
                evidence = self._bm25_retrieve(question, doc_sentences)
                all_qa_inputs.append(
                    {"question": question, "context": evidence or texts[i]}
                )
            sent_qa_slices.append((qa_start, len(all_qa_inputs)))

        # ── Phase 5: batch Extractive QA ────────────────────────────────────
        logger.info(f"Decontextualizer: QA batch — {len(all_qa_inputs)} inputs...")
        if all_qa_inputs:
            raw_qa = self.qa_pipe(all_qa_inputs, batch_size=DECONTEXT_GEN_BATCH_SIZE)
            all_qa_results: List[dict] = (
                raw_qa if isinstance(raw_qa, list) else [raw_qa]
            )
        else:
            all_qa_results = []

        # ── Phase 6: build QA2D prompt list ─────────────────────────────────
        all_qa2d_prompts: List[str]   = []
        sent_qa2d_slices: List[tuple] = []   # (start, end) into all_qa2d_prompts

        for (qg_start, qg_end), (qa_start, qa_end) in zip(
            sent_qg_slices, sent_qa_slices
        ):
            qa2d_start = len(all_qa2d_prompts)
            for q, r in zip(
                all_questions[qg_start:qg_end], all_qa_results[qa_start:qa_end]
            ):
                if r["score"] > QA_SCORE_THRESHOLD:
                    all_qa2d_prompts.append(
                        f"Convert to a declarative sentence: Q: {q} A: {r['answer']}"
                    )
            sent_qa2d_slices.append((qa2d_start, len(all_qa2d_prompts)))

        # ── Phase 7: batch QA-to-Declarative ────────────────────────────────
        logger.info(
            f"Decontextualizer: QA2D batch — {len(all_qa2d_prompts)} prompts..."
        )
        all_qa2d_results = self._generate_batch(
            all_qa2d_prompts, self.gen_tokenizer, self.gen_model,
            batch_size=DECONTEXT_GEN_BATCH_SIZE,
        )

        # ── Phase 8: build final rewrite prompt list ─────────────────────────
        all_rewrite_prompts: List[str]        = []
        sent_rewrite_idx: List[Optional[int]] = []  # index into all_rewrite_prompts, or None

        for i, (qa2d_start, qa2d_end) in enumerate(sent_qa2d_slices):
            declarative = [
                s for s in all_qa2d_results[qa2d_start:qa2d_end]
                if s and not s.rstrip().endswith("?")
            ]
            if declarative and texts[i]:
                full_context = " ".join(declarative)
                final_prompt = (
                    f"Rewrite the sentence to be self-contained by incorporating "
                    f"specific details from the context.\n"
                    f"Context: {full_context}\n"
                    f"Sentence: {texts[i]}\n"
                    f"Rewrite:"
                )
                sent_rewrite_idx.append(len(all_rewrite_prompts))
                all_rewrite_prompts.append(final_prompt)
            else:
                sent_rewrite_idx.append(None)

        # ── Phase 9: batch final rewrite ────────────────────────────────────
        logger.info(
            f"Decontextualizer: Rewrite batch — {len(all_rewrite_prompts)} prompts..."
        )
        all_rewrites = self._generate_batch(
            all_rewrite_prompts, self.gen_tokenizer, self.gen_model,
            batch_size=DECONTEXT_GEN_BATCH_SIZE,
        )

        # ── Phase 10: apply results back to SentenceScore objects ───────────
        for i, sent_obj in enumerate(sentences):
            text = texts[i]
            if not text:
                sent_obj.text = ""
                continue

            rw_idx = sent_rewrite_idx[i]
            if rw_idx is not None:
                rewritten = all_rewrites[rw_idx]
                max_len   = int(len(text) * DECONTEXT_REWRITE_RATIO)
                if (
                    rewritten
                    and rewritten.lower() != text.lower()
                    and "?" not in rewritten
                    and len(rewritten) <= max_len
                ):
                    sent_obj.original_text = text
                    sent_obj.text          = rewritten
                else:
                    sent_obj.text = text
            else:
                sent_obj.text = text

        logger.info("Decontextualizer: Complete.")
        return sentences

if RUN_DEBUG_STEPS:
    print("--- Running Decontextualizer Test ---")
    try:
        clean_result  = NLPResult()
        clean_options = NLPOptions(min_confidence=0.8, max_claims=10)

        pre_tmp   = Preprocessor()
        sents_tmp = pre_tmp.run(article, clean_result, clean_options)

        ner_tmp = EntityRecognizer()
        ner_tmp.run(article, clean_result, clean_options, sents_tmp)

        ext_tmp   = SentenceExtraction(use_fp16=True)
        sents_tmp = ext_tmp.run(article, clean_result, clean_options, sents_tmp)

        print(f"\n{len(sents_tmp)} sentences after extraction. Running Decontextualizer...")

        decon = Decontextualizer(use_gpu=True)
        old_texts = [s.text for s in sents_tmp]
        sents_tmp = decon.run(article, clean_result, clean_options, sents_tmp)

        print("\nDecontextualization Results:")
        for old, new_obj in zip(old_texts, sents_tmp):
            print(f"  Original : {old}")
            print(f"  Rewritten: {new_obj.text}")
            print("  " + "-" * 60)

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()


In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Decontextualizer Test ---")
    try:
        # Rebuild a clean local sentences list from scratch
        clean_result  = NLPResult()
        clean_options = NLPOptions(min_confidence=0.8, max_claims=10)

        pre_tmp  = Preprocessor()
        sents_tmp = pre_tmp.run(article, clean_result, clean_options)

        ner_tmp  = EntityRecognizer()
        ner_tmp.run(article, clean_result, clean_options, sents_tmp)

        ext_tmp  = SentenceExtraction(use_fp16=True)
        sents_tmp = ext_tmp.run(article, clean_result, clean_options, sents_tmp)

        print(f"\n{len(sents_tmp)} sentences after extraction. Running Decontextualizer...")

        decon = Decontextualizer(use_gpu=True)
        old_texts  = [s.text for s in sents_tmp]
        sents_tmp  = decon.run(article, clean_result, clean_options, sents_tmp)

        print("\nDecontextualization Results:")
        for old, new_obj in zip(old_texts, sents_tmp):
            print(f"  Original : {old}")
            print(f"  Rewritten: {new_obj.text}")
            print("  " + "-" * 60)

    except Exception as e:
        import traceback
        print(f"Error: {e}")
        traceback.print_exc()


__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Running Decontextualizer Test ---


__main__ - INFO - Preprocessor: Cleaned & Split. Result: 26 sentences.
__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...
Loading weights: 100%|██████████| 199/199 [00:01<00:00, 145.62it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER-uncased
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
__main__ - INFO - EntityRecognizer: Running NER on 26 sentences.
__main__ - INFO - EntityRecognizer: Found 20 unique entities.
__main__ - INFO - SentenceExtraction: Initializing models on cuda (fp16=True)...
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 261.34it/s, Materializing param=pooler.dense.weight] 

## Checkworthy

In [ ]:
import logging
import torch
import spacy
from typing import List
from transformers import pipeline

# Local imports
from microservices.nlp.models.base import SentenceProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore
from microservices.nlp.config import (
    CW_THRESHOLD, CW_BATCH_SIZE,
    CW_NLI_MODEL, CW_NLI_WEIGHT, CW_HEURISTIC_WEIGHT,
)

logger = logging.getLogger(__name__)

# ── Zero-shot candidate labels ────────────────────────────────────────────────
_CW_LABELS     = ["verifiable factual claim", "opinion or speculation", "question or request"]
_FACTUAL_LABEL = "verifiable factual claim"

# ── Named-entity label sets ───────────────────────────────────────────────────
_NE_LABELS      = {"PERSON", "ORG", "GPE", "EVENT", "FAC", "NORP", "LOC", "PRODUCT"}
_NUMERIC_LABELS = {"MONEY", "PERCENT", "CARDINAL", "DATE", "QUANTITY", "TIME", "ORDINAL"}

# ── Reporting verbs (extended) ────────────────────────────────────────────────
_REPORTING_VERBS = {
    "say", "claim", "state", "report", "announce", "confirm",
    "warn", "accuse", "allege", "assert", "declare", "deny",
    "reveal", "disclose", "admit", "acknowledge",
}

# ── Hyland (1998) hedging lexicon ─────────────────────────────────────────────
_HEDGE_LEMMAS = {
    "may", "might", "could", "would", "should",
    "seem", "appear", "suggest", "indicate", "assume", "believe",
    "think", "suppose", "guess", "expect", "predict", "estimate",
    "possibly", "probably", "perhaps", "likely", "unlikely",
    "approximately", "roughly", "around", "about", "allegedly",
    "reportedly", "supposedly", "potentially", "conceivably",
    "if", "unless", "whether",
}


class CheckWorthiness(SentenceProcessor):
    """
    MODULAR CHECK-WORTHINESS LAYER — HYBRID TRANSFORMER + LEXICAL

    Stage 1: Zero-Shot NLI Transformer (CW_NLI_MODEL)
      Classifies each sentence as "verifiable factual claim", "opinion or speculation",
      or "question or request". The entailment probability for "verifiable factual claim"
      is the transformer score.

    Stage 2: Enhanced spaCy Heuristic
      SVO triple detection via dependency parse, question suppression,
      extended NER label sets, and Hyland (1998) hedging lexicon.

    Stage 3: Ensemble
      final_score = CW_NLI_WEIGHT * nli_score + CW_HEURISTIC_WEIGHT * heuristic_score

    Accepts and returns a local sentences list; does NOT write to result.
    """

    def __init__(self, nlp=None):
        device_id = 0 if torch.cuda.is_available() else -1
        dtype     = torch.float16 if torch.cuda.is_available() else torch.float32

        if nlp is not None:
            logger.info("CheckWorthiness: Using shared spaCy model.")
            self.nlp = nlp
        else:
            logger.info("CheckWorthiness: Loading spaCy 'en_core_web_sm'...")
            try:
                self.nlp = spacy.load("en_core_web_sm")
            except OSError:
                logger.error("CheckWorthiness: Run: python -m spacy download en_core_web_sm")
                raise

        logger.info(
            f"CheckWorthiness: Loading NLI classifier ({CW_NLI_MODEL}) "
            f"on {'CUDA' if device_id == 0 else 'CPU'} (fp16={device_id == 0})..."
        )
        try:
            self.nli_classifier = pipeline(
                "zero-shot-classification",
                model=CW_NLI_MODEL,
                device=device_id,
                torch_dtype=dtype,
            )
            logger.info("CheckWorthiness: NLI classifier loaded.")
        except Exception as e:
            logger.error(f"CheckWorthiness: Failed to load NLI classifier: {e}")
            raise

    def _heuristic_score(self, doc) -> float:
        if doc.text.strip().endswith("?"):
            return 0.0

        has_subject = any(t.dep_ in ("nsubj", "nsubjpass") for t in doc)
        if not has_subject:
            return 0.1

        has_svo = False
        for token in doc:
            if token.dep_ == "ROOT" and token.pos_ == "VERB":
                children_deps = {c.dep_ for c in token.children}
                if "nsubj" in children_deps and children_deps & {"dobj", "attr", "pobj", "nsubjpass"}:
                    has_svo = True
                    break

        named_ents   = [e for e in doc.ents if e.label_ in _NE_LABELS]
        numeric_ents = [e for e in doc.ents if e.label_ in _NUMERIC_LABELS]

        has_reporting = any(
            t.pos_ == "VERB" and t.lemma_.lower() in _REPORTING_VERBS
            for t in doc
        )

        hedge_count   = sum(1 for t in doc if t.lemma_.lower() in _HEDGE_LEMMAS)
        hedge_penalty = min(0.40, hedge_count * 0.15)

        score = 0.0
        if has_svo:                score += 0.40
        if len(named_ents) >= 1:   score += 0.20
        if len(named_ents) >= 2:   score += 0.10
        if len(numeric_ents) >= 1: score += 0.25
        if has_reporting:          score += 0.10
        score -= hedge_penalty

        return max(0.0, min(1.0, score))

    def _run_nli_batch(self, texts: List[str]) -> List[float]:
        if not texts:
            return []
        results = self.nli_classifier(
            texts,
            _CW_LABELS,
            multi_label=False,
            hypothesis_template="This sentence is a {}.",
        )
        scores = []
        for res in results:
            label_scores = dict(zip(res["labels"], res["scores"]))
            scores.append(label_scores.get(_FACTUAL_LABEL, 0.0))
        return scores

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Scores each sentence using the hybrid NLI + heuristic ensemble.
        Populates confidence and is_checkworthy on each SentenceScore in-place.
        Returns the same list; does NOT write to result.
        """
        if not sentences:
            return []

        logger.info(
            f"CheckWorthiness: Evaluating {len(sentences)} sentences "
            f"(batch_size={CW_BATCH_SIZE}, threshold={CW_THRESHOLD}, "
            f"nli_weight={CW_NLI_WEIGHT}, heuristic_weight={CW_HEURISTIC_WEIGHT})..."
        )

        texts = [s.text for s in sentences]

        spacy_docs = list(self.nlp.pipe(texts, batch_size=CW_BATCH_SIZE))
        nli_scores = self._run_nli_batch(texts)

        for s_obj, doc, nli_score in zip(sentences, spacy_docs, nli_scores):
            heuristic   = self._heuristic_score(doc)
            final_score = CW_NLI_WEIGHT * nli_score + CW_HEURISTIC_WEIGHT * heuristic
            final_score = max(0.0, min(1.0, final_score))
            s_obj.confidence     = float(final_score)
            s_obj.is_checkworthy = final_score >= CW_THRESHOLD

        worthy_count = sum(1 for s in sentences if s.is_checkworthy)
        logger.info(
            f"CheckWorthiness: {worthy_count}/{len(sentences)} sentences marked as check-worthy."
        )
        return sentences

if RUN_DEBUG_STEPS:
    print("--- Running Check-Worthiness Test ---")
    try:
        cw = CheckWorthiness()
        sentences = cw.run(article, result, options, sentences)

        print("\nCheck-Worthiness Scores:")
        for s in sentences:
            status = "[CHECK]" if s.is_checkworthy else "[IGNORE]"
            print(f"{status} Score: {s.confidence:.2f} | {s.text[:80]}...")

    except Exception as e:
        print(f"Error: {e}")

In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Check-Worthiness Test ---")
    try:
        cw = CheckWorthiness()
        sentences = cw.run(article, result, options, sentences)
        
        print("\nCheck-Worthiness Scores:")
        for s in sentences:
            status = "[CHECK]" if s.is_checkworthy else "[IGNORE]"
            print(f"{status} Score: {s.confidence:.2f} | {s.text[:80]}...")

    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - CheckWorthiness: Loading Spacy model...


--- Running Check-Worthiness Test ---


__main__ - INFO - CheckWorthiness: Evaluating 10 sentences (batch_size=32)...
__main__ - INFO - CheckWorthiness: Scoring complete.



Check-Worthiness Scores:
[CHECK] Score: 0.90 | FBI raids Georgia election office over 2020 voter fraud claims....
[CHECK] Score: 0.90 | The FBI raided a Georgia election office on Wednesday as it examined allegations...
[IGNORE] Score: 0.50 | "This is an assault on your vote," Fulton County Commissioner Mo Ivory said at a...
[CHECK] Score: 0.90 | The 2020 election marked the first time since 1992 that a Democrat had won the s...
[CHECK] Score: 0.90 | The state of Georgia, and Fulton County in particular, were a major focus of Tru...
[IGNORE] Score: 0.40 | Numerous courts rejected legal challenges and claims made by Trump and his allie...
[IGNORE] Score: 0.50 | Raffensperger, whose office oversees Georgia's elections and certifies results, ...
[CHECK] Score: 0.90 | Trump faced two criminal indictments related to alleged election interference in...
[CHECK] Score: 0.90 | Trump officials sue Georgia county to force release of 2020 voting records....
[IGNORE] Score: 0.50 | WW1 toxic compou

In [ ]:
import logging
import torch
import numpy as np
from typing import List
from sentence_transformers import SentenceTransformer

# Local imports
from microservices.nlp.models.base import SentenceProcessor
from common.models.api.redis_models import Article, NLPOptions, NLPResult, SentenceScore
from microservices.nlp.config import EMBEDDING_MODEL, EMBEDDER_BATCH_SIZE

logger = logging.getLogger(__name__)


class Embedder(SentenceProcessor):
    """
    MODULAR EMBEDDER LAYER

    Generates 768-dimensional dense vector embeddings for every extracted
    sentence using EMBEDDING_MODEL (default: sentence-transformers/all-mpnet-base-v2).

    FP16 on CUDA; torch.inference_mode() for all forward passes.
    Accepts and returns a local sentences list (embedding field populated in-place);
    does NOT otherwise modify result.
    """

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        use_fp16 = torch.cuda.is_available()

        logger.info(f"Embedder: Loading '{self.model_name}' on {self.device} (fp16={use_fp16})...")
        try:
            self.model = SentenceTransformer(self.model_name, device=self.device)
            if use_fp16:
                self.model.half()
            logger.info("Embedder: Model loaded successfully.")
        except Exception as e:
            logger.error(f"Embedder: Failed to load model: {e}")
            raise

    def run(self, article: Article, result: NLPResult, options: NLPOptions,
            sentences: List[SentenceScore]) -> List[SentenceScore]:
        """
        Generates embeddings for every sentence in the local list.
        Updates each SentenceScore.embedding in-place.
        Returns the same list; does NOT modify result.
        """
        if not sentences:
            logger.info("Embedder: No sentences to process.")
            return []

        texts = [s.text for s in sentences]

        try:
            with torch.inference_mode():
                embeddings: np.ndarray = self.model.encode(
                    texts,
                    batch_size=EMBEDDER_BATCH_SIZE,
                    show_progress_bar=len(texts) > EMBEDDER_BATCH_SIZE,
                    convert_to_numpy=True,
                    normalize_embeddings=False,
                )

            for i, sent in enumerate(sentences):
                sent.embedding = embeddings[i].tolist()

            logger.info(f"Embedder: Vectorized {len(texts)} sentences ({embeddings.shape[1]}-dim).")

        except Exception as e:
            logger.error(f"Embedder: Encoding failed: {e}")
            raise

        return sentences

if RUN_DEBUG_STEPS:
    print("--- Running Embedder Test ---")
    try:
        emb = Embedder()
        sentences = emb.run(article, result, options, sentences)

        print(f"Successfully vectorized {len(sentences)} sentences.")
        if sentences:
            print(f"Sample Embedding (First 3 dims): {sentences[0].embedding[:3]}...")

    except Exception as e:
        print(f"Error: {e}")

In [ ]:
import logging
import time
from typing import List
from microservices.nlp.models.base import NLPComponent
from common.models.api.redis_models import Article, NLPOptions, NLPResult, Claim, SentenceScore

logger = logging.getLogger(__name__)

class ClaimExtraction(NLPComponent):
    """
    THE ORCHESTRATOR
    Threads a local List[SentenceScore] through every stage.
    Only writes to result at the very end (claims_in_article, entities_in_article).
    """
    def __init__(self, use_gpu: bool = True):
        logger.info("ClaimExtraction: Initializing pipeline orchestrator...")
        self.preprocessor       = Preprocessor()
        self.entity_recognizer  = EntityRecognizer()
        self.sentence_extractor = SentenceExtraction()
        self.decontextualizer   = Decontextualizer(use_gpu=use_gpu)
        self.checkworthiness    = CheckWorthiness()
        self.final_embedder     = Embedder()

    def _map_entities_to_sentences(self, sentences: List[SentenceScore], result: NLPResult):
        """Maps global entities onto each sentence by text overlap."""
        if not sentences or not result.entities_in_article:
            return
        for s_obj in sentences:
            s_obj.entities = [
                ent for ent in result.entities_in_article
                if ent.entity_text.lower() in s_obj.text.lower()
            ]

    def run(self, article: Article, result: NLPResult, options: NLPOptions) -> None:
        total_start = time.time()

        # Stage 1 — Preprocessing → local list
        t = time.time()
        sentences = self.preprocessor.run(article, result, options)
        logger.info(f"Stage 1 [Preprocessor] complete in {time.time()-t:.2f}s — {len(sentences)} sentences")

        # Stage 2 — NER → writes to result.entities_in_article only
        t = time.time()
        self.entity_recognizer.run(article, result, options, sentences)
        logger.info(f"Stage 2 [NER] complete in {time.time()-t:.2f}s — {len(result.entities_in_article)} entities")

        # Stage 3 — Extraction + deduplication → filtered local list
        t = time.time()
        sentences = self.sentence_extractor.run(article, result, options, sentences)
        logger.info(f"Stage 3 [SentenceExtraction] complete in {time.time()-t:.2f}s — {len(sentences)} kept")

        # Stage 4 — Decontextualization → rewrites texts in local list
        t = time.time()
        sentences = self.decontextualizer.run(article, result, options, sentences)
        logger.info(f"Stage 4 [Decontextualizer] complete in {time.time()-t:.2f}s")

        # Stage 5 — Check-worthiness scoring
        t = time.time()
        sentences = self.checkworthiness.run(article, result, options, sentences)
        logger.info(f"Stage 5 [CheckWorthiness] complete in {time.time()-t:.2f}s")

        # Stage 5.5 — Map entities onto sentences
        t = time.time()
        self._map_entities_to_sentences(sentences, result)
        logger.info(f"Stage 5.5 [Entity Mapping] complete in {time.time()-t:.2f}s")

        # Stage 6 — Embed sentences
        t = time.time()
        sentences = self.final_embedder.run(article, result, options, sentences)
        logger.info(f"Stage 6 [Embedder] complete in {time.time()-t:.2f}s")

        # Stage 7 — Convert local SentenceScore list → result.claims_in_article
        t = time.time()
        result.claims_in_article = [
            Claim(
                confidence=s.confidence,
                source_sentence_indices=[s.index],
                decontextualised_claim_text=s.text,
                decontextualised_claim_embedding=s.embedding,
                NER_entities=getattr(s, "entities", []),
            )
            for s in sentences
        ]
        logger.info(
            f"Stage 7 [Sentence→Claim] complete in {time.time()-t:.2f}s — "
            f"{len(result.claims_in_article)} claims stored"
        )

        logger.info(f"--- Pipeline Finished in {time.time()-total_start:.2f}s ---")


In [ ]:
if RUN_DEBUG_STEPS:
    print("--- Running Embedder Test ---")
    try:
        emb = Embedder()
        sentences = emb.run(article, result, options, sentences)
        
        print(f"Successfully vectorized {len(sentences)} sentences.")
        if sentences:
            print(f"Sample Embedding (First 3 dims): {sentences[0].embedding[:3]}...")
            
    except Exception as e:
        print(f"Error: {e}")


__main__ - INFO - Embedder: Loading sentence-transformers/all-mpnet-base-v2 on cuda (fp16=True)...
sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-mpnet-base-v2


--- Running Embedder Test ---


httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/mod

Successfully vectorized 10 sentences.
Sample Embedding (First 3 dims): [0.04150390625, 0.052764892578125, 0.0225982666015625]...


: 

In [ ]:
if RUN_DEBUG_STEPS:
    import time
    import json
    import dataclasses

    print("--- Loading Article Data ---")

    final_result  = NLPResult()
    final_options = NLPOptions(
        min_confidence=0.8,
        max_claims=5,
    )

    print("\n--- Initializing ClaimExtraction Orchestrator ---")
    start_init = time.time()
    pipeline_service = ClaimExtraction(use_gpu=True)
    print(f"Orchestrator ready in {time.time() - start_init:.2f}s")

    print("\n" + "="*80)
    print(f"RUNNING PIPELINE: {article.title}")
    print("="*80)

    try:
        pipeline_service.run(article, final_result, final_options)
    except Exception as e:
        logger.error(f"Pipeline failed at runtime: {e}")

    print("\n" + "="*80)
    print(f"{'FINAL REFINED CLAIMS':^80}")
    print("="*80)

    if not final_result.claims_in_article:
        print("No claims were extracted from this article.")
    else:
        for i, claim in enumerate(final_result.claims_in_article, 1):
            print(f"\nClaim #{i}")
            print(f"Confidence: {claim.confidence:.2f}")
            print(f"Source Sentence Index: {claim.source_sentence_indices}")
            print(f"Text: {claim.decontextualised_claim_text}")
            if claim.NER_entities:
                entity_names = [f"{e.entity_text} ({e.type_of_entity})" for e in claim.NER_entities]
                print(f"Entities: {', '.join(entity_names)}")
            if claim.decontextualised_claim_embedding:
                print(f"Vector: [Stored - {len(claim.decontextualised_claim_embedding)} dimensions]")

    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    print(f"Total Claims Extracted : {len(final_result.claims_in_article)}")
    print(f"Total Unique Entities  : {len(final_result.entities_in_article)}")

    output_file = 'final_claims_output.json'
    with open(output_file, 'w') as f:
        json.dump(dataclasses.asdict(final_result), f, indent=2, default=str)
    print(f"\n✓ Full structured data saved to: {output_file}")


__main__ - INFO - ClaimExtraction: Initializing pipeline orchestrator...
__main__ - INFO - Preprocessor: Loading Spacy 'en_core_web_sm' model...


--- Loading Article Data ---

--- Initializing ClaimExtraction Orchestrator ---


__main__ - INFO - EntityRecognizer: Loading 'dslim/bert-base-NER-uncased' on CUDA (fp16=True)...
httpx - INFO - HTTP Request: HEAD https://huggingface.co/dslim/bert-base-NER-uncased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dslim/bert-base-NER-uncased/301540b48433e31000058882c971ddd7bc726547/config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/dslim/bert-base-NER-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/dslim/bert-base-NER-uncased/301540b48433e31000058882c971ddd7bc726547/tokenizer_config.json "HTTP/1.1 200 OK"
httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/dslim/bert-base-NER-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
httpx - INFO - HTTP Request: GET https://hugging